# ESA SpOC-4 — Matching I & Matching II | Kaggle

This notebook runs the beginner 3D matching solvers, validates their selected routes and creates the ESA JSON files. It **does not solve orbital Trajectory Matching**. You must attach the official ESA data files, and the ESA website—not Kaggle—validates the submissions.

## A. Attach input files

1. Add the starter ZIP as a Kaggle Dataset via **Add Input** (or use your own public GitHub repository).
2. Add the official matching datasets (from ESA About → Instance Files). Expected row counts: 25,000 and 92,103.
3. Choose **CPU** / Accelerator **None**. This mixed integer programming solver runs on CPU.

In [ ]:
from pathlib import Path
import os, sys, zipfile, shutil, subprocess
print('Kaggle input directories:')
for p in sorted(Path('/kaggle/input').glob('*')):
    print(' -',p)
print('Current working directory:',os.getcwd())

In [ ]:
# Find the provided project ZIP and extract it. You can override ZIP_FILE manually.
ZIP_FILE = None   # e.g. '/kaggle/input/my-luna-project/Luna-Tomato-Logistics-Kaggle-GitHub.zip'
if ZIP_FILE is None:
    candidates=[p for p in Path('/kaggle/input').rglob('*.zip') if 'luna-tomato-logistics' in p.name.lower()]
    ZIP_FILE=str(candidates[0]) if candidates else None
WORK=Path('/kaggle/working')
ROOT=WORK/'Luna-Tomato-Logistics'
if ZIP_FILE:
    print('Using', ZIP_FILE)
    with zipfile.ZipFile(ZIP_FILE) as z: z.extractall(WORK)
else:
    # Alternative: set REPO_URL to your real public GitHub repository.
    REPO_URL = ''  # e.g. 'https://github.com/YOUR_USERNAME/ESA-SpOC4-Luna-Tomato-Logistics.git'
    if REPO_URL and not ROOT.exists():
        subprocess.run(['git','clone',REPO_URL,str(ROOT)],check=True)
if not (ROOT/'src'/'cli.py').exists():
    raise FileNotFoundError('Starter project not found. Attach its ZIP or configure REPO_URL above.')
os.chdir(ROOT)
print('Loaded project:',ROOT)

In [ ]:
# Kaggle's Python commonly already includes scipy/numpy; install only if missing.
try:
    import scipy, numpy, pandas, matplotlib
    print('SciPy', scipy.__version__)
except ImportError:
    subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'],check=True)
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],check=True)

In [ ]:
# Identify files by *exact row count* so their names may differ in ESA downloads.
# Only non-comment lines with 4 whitespace-separated numeric fields count.
def matching_rows(path):
    count=0
    try:
        with open(path,encoding='utf-8') as f:
            for line in f:
                parts=line.split()
                if not parts or parts[0].startswith('#'): continue
                if len(parts)!=4:return -1
                for x in parts:float(x)
                count+=1
        return count
    except (OSError, ValueError, UnicodeError):
        return -1

for problem, n in [('matching-i',25000),('matching-ii',92103)]:
    target=ROOT/'data'/f'{problem}.txt'
    if not target.exists():
        for p in Path('/kaggle/input').rglob('*.txt'):
            if matching_rows(p)==n:
                shutil.copyfile(p,target)
                print('Copied official-looking dataset',p,'to',target)
                break
    if target.exists():
        actual=matching_rows(target)
        print(problem, 'row count',actual,'expected',n)
        if actual!=n:raise ValueError(f'{target} has wrong row count')
    else:
        print('MISSING:',target,'— add the ESA file to Kaggle Input')

## B. Run official-sized problems

Start with a feasible heuristic for a baseline. Then run hybrid optimization. Increase `seconds` and `restarts` for further attempts. An MILP time-limit result is not necessarily optimal.

In [ ]:
from time import perf_counter
RUNS=[('matching-i',300,20),('matching-ii',600,30)]
for problem,seconds,restarts in RUNS:
    source=ROOT/'data'/f'{problem}.txt'
    if not source.exists():
        print('Skipped (missing ESA data):',problem)
        continue
    command=[sys.executable,'-m','src.cli','--input',str(source),'--problem',problem,
             '--method','hybrid','--seconds',str(seconds),'--restarts',str(restarts),
             '--seed','2026','--out',str(ROOT/'results')]
    print('Running',problem,'with',seconds,'seconds MILP budget')
    subprocess.run(command,check=True,cwd=ROOT)

In [ ]:
# Summarize completed ESA-sized results and graph the positive local masses.
import json
import pandas as pd
import matplotlib.pyplot as plt
rows=[]
for problem in ('matching-i','matching-ii'):
    report=ROOT/'results'/f'report_{problem}.json'
    if report.exists():
        info=json.loads(report.read_text())
        if info['rows'] not in (25000,92103):
            print('Skipping a demo result:',problem)
            continue
        rows.append({'problem':problem,'local_mass':info['score_local'],
                     'ESA_fitness_expected':info['esa_fitness_expected'],
                     'selected':info['n_selected'],
                     'milp_gap':info.get('milp',{}).get('mip_gap')})
if rows:
    df=pd.DataFrame(rows)
    display(df)
    ax=df.plot(kind='bar',x='problem',y='local_mass',legend=False,title='Delivered mass (local objective)')
    ax.set_ylabel('Mass (input weight units)')
    plt.tight_layout()
    plt.savefig(ROOT/'results'/'comparison.png',dpi=160)
    plt.show()
else:
    print('No ESA-sized results yet. Attach the official data and rerun.')

## C. Submit

Under the notebook **Output** (after Save Version → Save & Run All), download `submission_matching-i.json` and `submission_matching-ii.json`. Upload each separately to the corresponding ESA Optimise problem page. Validate against ESA's official evaluator. GitHub/Kaggle rankings are not ESA rankings.